In [47]:
import json
import sys
from rich import print as rprint
from pathlib import Path
import re

nb_dir = Path.cwd()

project_root = nb_dir.parent.parent

sys.path.insert(0, str(project_root))


from scripts.text_matching import normalise_text

In [48]:
unmatched_file = Path(project_root / "data/people/people_unmatched.json")
people_file = Path(project_root / "data/from db/people.json")
variants_file = Path(project_root / "data/from db/people_variants.json")

In [49]:
with open(unmatched_file , "r") as f:
   unmatched = json.load(f)
with open(people_file , "r") as f:
   people = json.load(f)
with open(variants_file , "r") as f:
   variants = json.load(f)

rprint(people[:3])
# rprint(dict(list(unmatched.items())[:3]))
rprint(variants[:5])

[
    {
        'person_id': 1,
        'unified_id': 'abel_othenio',
        'family_name': 'Abel',
        'given_names': 'Othenio',
        'name_particles': None,
        'single_name': None,
        'is_organisation': False,
        'created_at': '2026-01-26 19:54:23.311152',
        'updated_at': '2026-01-26 19:54:23.311152'
    },
    {
        'person_id': 2,
        'unified_id': 'abel_otto',
        'family_name': 'Abel',
        'given_names': 'Otto',
        'name_particles': None,
        'single_name': None,
        'is_organisation': False,
        'created_at': '2026-01-26 19:54:23.311152',
        'updated_at': '2026-01-26 19:54:23.311152'
    },
    {
        'person_id': 3,
        'unified_id': 'absolon_kurt',
        'family_name': 'Absolon',
        'given_names': 'Kurt',
        'name_particles': None,
        'single_name': None,
        'is_organisation': False,
        'created_at': '2026-01-26 19:54:23.311152',
        'updated_at': '2026-01-26 19:54:23.311152'
    }
]

[
    {
        'variant_id': 394,
        'person_id': 1,
        'unified_id': 'abel_othenio',
        'variant_string': 'Abel, Othenio',
        'variant_normalised': 'abel, othenio',
        'source': 'prelim',
        'created_at': '2026-04-24 15:45:23.652127'
    },
    {
        'variant_id': 395,
        'person_id': 2,
        'unified_id': 'abel_otto',
        'variant_string': 'Abel, Otto',
        'variant_normalised': 'abel, otto',
        'source': 'prelim',
        'created_at': '2026-04-24 15:45:23.652127'
    },
    {
        'variant_id': 396,
        'person_id': 2,
        'unified_id': 'abel_otto',
        'variant_string': 'Otto Abel u. Wilhelm Wattenbach',
        'variant_normalised': 'otto abel u. wilhelm wattenbach',
        'source': 'prelim',
        'created_at': '2026-04-24 15:45:23.652127'
    },
    {
        'variant_id': 397,
        'person_id': 3,
        'unified_id': 'absolon_kurt',
        'variant_string': 'Kurt Absolon',
        'variant_normalised': 'kurt absolon',
        'source': 'prelim',
        'created_at': '2026-04-24 15:45:23.652127'
    },
    {
        'variant_id': 398,
        'person_id': 4,
        'unified_id': 'abu_temmam',
        'variant_string': 'Abu Temmam',
        'variant_normalised': 'abu temmam',
        'source': 'prelim',
        'created_at': '2026-04-24 15:45:23.652127'
    }
]

In [ ]:
people_dict = {}

for person in people:
    last = person["family_name"]
    first = person["given_names"]

    if last is None or first is None:
        continue

    last_norm = normalise_text(last)
    first_norm = normalise_text(first)

    key = (last_norm, first_norm)
    people_dict[key] = person["person_id"]

# rprint(dict(list(people_dict.items())[:5]))

single_dict = {}

for person in people:
    single = person["single_name"]
    if not single:
        continue
    else:
        single_norm = normalise_text(single)

    single_dict[single_norm] = person["person_id"]

# rprint(dict(list(single_dict.items())[:5]))
# rprint(len(single_dict))

variants_dict = {}


{
    ('abel', 'othenio'): 1,
    ('abel', 'otto'): 2,
    ('absolon', 'kurt'): 3,
    ('abulafia', 'david'): 5,
    ('achelis', 'joh. daniel'): 6
}

{'abu temmam': 4, 'aischylos': 40, 'al ghasali': 45, 'al-halladsch': 46, 'al-hamadhani': 47}

295

In [ ]:
matched_now = {}
still_unmatched = {}
comma = ", "

for name, entries in unmatched.items():
    if (re.search(comma, name)):
        names_split = re.split(comma, name)
        first = names_split[1]
        last = names_split[0]

    else:
        names_split = name.rsplit(" ", 1)
        if len(names_split) < 2:
            person_id = single_dict.get(name)
            if person_id:
                matched_now[person_id] = entries
            else:
                still_unmatched[name] = entries
            continue
        else:
            first = names_split[0]
            last = names_split[1]
    # rprint(names_split)
    person_id = people_dict.get((last, first))


    if person_id:
        matched_now[person_id] = entries
    else:
        still_unmatched[name] = entries

print(len(matched_now))
print(len(still_unmatched))
# rprint(dict(list(matched_now.items())[:5]))
# rprint(dict(list(still_unmatched.items())[:5]))
with open("matched_now.json", "w") as f:
    json.dump(matched_now, f, ensure_ascii=False, indent=2)

with open("still_unmatched.json", "w") as f:
    json.dump(still_unmatched, f, ensure_ascii=False, indent=2)


1856
1609
